In [6]:
import pandas as pd
import numpy as np
import re

from tensorflow import keras
from tensorflow.keras import layers
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

df = pd.read_csv("IMDB Dataset.csv")

df['review'] = df['review'].str.lower()

stop_words = set(ENGLISH_STOP_WORDS) - {"not", "no", "nor", "never"}

df['review'] = df['review'].apply(
    lambda x: " ".join(
        word for word in x.split()
        if word not in stop_words
    )
)

df['sentiment'] = df['sentiment'].map({
    "positive":1,
    "negative":0
})

train, test = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

train_words = [review.split() for review in train['review']]
test_words = [review.split() for review in test['review']]

w2v = Word2Vec(
    sentences=train_words,
    window=2,
    min_count=2,
    vector_size=50,
    sg=0,
    epochs=5
)

def document_vectors(words):
    vectors = [w2v.wv[word] for word in words if word in w2v.wv]

    if len(vectors) > 0:
        return np.mean(vectors, axis=0)

    return np.zeros(50)


x_train = np.array([document_vectors(x) for x in train_words])
x_test = np.array([document_vectors(x) for x in test_words])

y_train = train['sentiment'].values
y_test = test['sentiment'].values

# 11. Print shapes
print("X_train:", x_train.shape)
print("Y_train:", y_train.shape)

print("X_test:", x_test.shape)
print("Y_test:", y_test.shape)

model = keras.Sequential([
    layers.Input(shape=(50,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=['accuracy']
)

history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=32
)

# 15. Evaluate model
loss, accuracy = model.evaluate(x_test, y_test)

print("Loss:", loss)
print("Accuracy:", accuracy)



X_train: (40000, 50)
Y_train: (40000,)
X_test: (10000, 50)
Y_test: (10000,)
Epoch 1/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.7637 - loss: 0.5021
Epoch 2/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7976 - loss: 0.4408
Epoch 3/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.8005 - loss: 0.4355
Epoch 4/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.8018 - loss: 0.4327
Epoch 5/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.8041 - loss: 0.4300
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8062 - loss: 0.4291
Loss: 0.4290623962879181
Accuracy: 0.8062000274658203
